# EfficientNetV2-S + Semi-Supervised Learning (Pseudo-Labeling)

**Pipeline overview:**
1. **Phase 1** — Supervised training on `train2` with camera-specific augmentations
2. **Phase 2** — Generate pseudo-labels for unlabeled test images (high-confidence only)
3. **Phase 3** — Iterative SSL fine-tuning on `train2 + pseudo-labeled test` (2 rounds, decreasing threshold)
4. **Final inference** — Submission CSV

Camera augmentations simulate the visual appearance of test cameras from train camera views.
Horizontal-flip augmentations correctly remap `Lateral_lying_left ↔ Lateral_lying_right`.


In [ ]:
# ── Installation ────────────────────────────────────────────────────────────
!pip install albumentations -q

# ── Kaggle credentials ──────────────────────────────────────────────────────
from google.colab import files
files.upload()   # upload kaggle.json

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# ── Download competition data ───────────────────────────────────────────────
!kaggle competitions download -c multi-view-pig-posture-recognition
!unzip -q multi-view-pig-posture-recognition.zip
!ls

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import os, ast, copy, re, time
from collections import Counter

import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset
from torchvision import transforms, models
import torchvision.transforms.functional as TF

from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# ── GPU setup ───────────────────────────────────────────────────────────────
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    _ = torch.zeros(1).cuda(); torch.cuda.synchronize()
    x = torch.randn(10000, 10000, device='cuda')
    t0 = time.time(); _ = x @ x; torch.cuda.synchronize()
    elapsed = time.time() - t0
    print(f'GPU test: {elapsed:.3f}s  {"✓ ok" if elapsed < 1.0 else "⚠ slow"}')
    del x; torch.cuda.empty_cache()

In [ ]:
# ── Constants ────────────────────────────────────────────────────────────────
CLASS_NAMES = {
    0: 'Lateral_lying_left',
    1: 'Lateral_lying_right',
    2: 'Sitting',
    3: 'Standing',
    4: 'Sternal_lying',
}
NUM_CLASSES = len(CLASS_NAMES)

BASE_DIR    = Path('multiview_pig_posture_recognition')
TRAIN2_IMGS = BASE_DIR / 'train2_images'
TEST_IMGS   = BASE_DIR / 'test_images'

# ── Hyperparameters ──────────────────────────────────────────────────────────
BATCH_SIZE        = 32
EPOCHS_PHASE1     = 10       # supervised training epochs
EPOCHS_SSL        = 5        # fine-tuning epochs per SSL round
SSL_ROUNDS        = 2        # number of pseudo-labeling iterations
SSL_CONF_THRESH   = [0.90, 0.85]   # confidence threshold per round
LR_PHASE1         = 1e-4
LR_SSL            = [5e-5, 2.5e-5] # learning rate per SSL round

SAVE_PATH_PHASE1  = 'effnetv2_phase1.pt'
SAVE_PATH_SSL     = 'effnetv2_ssl_final.pt'
SUBMISSION_PATH   = 'submission_ssl.csv'

print('Constants ✓')

In [ ]:
# ── Camera metadata parsing ──────────────────────────────────────────────────
def parse_camera_meta(image_id):
    m = re.match(r'(pen\d+)_(orb|tur)_(cam\d+)_', image_id)
    if m:
        return m.group(1), m.group(2), m.group(3)
    return 'unknown', 'unknown', 'unknown'

def add_camera_cols(df):
    df['pen']      = df['image_id'].apply(lambda x: parse_camera_meta(x)[0])
    df['cam_type'] = df['image_id'].apply(lambda x: parse_camera_meta(x)[1])
    df['cam_num']  = df['image_id'].apply(lambda x: parse_camera_meta(x)[2])
    df['camera']   = df['pen'] + '_' + df['cam_type'] + '_' + df['cam_num']
    return df

# ── Load train2 and test DataFrames ─────────────────────────────────────────
train2 = pd.read_csv(BASE_DIR / 'train2.csv')
train2['source']      = 'train2'
train2['bbox_parsed'] = train2['bbox'].apply(ast.literal_eval)
train2['class_name']  = train2['class_id'].map(CLASS_NAMES)
train2 = add_camera_cols(train2)

test = pd.read_csv(BASE_DIR / 'test.csv')
test['source']      = 'test'
test['bbox_parsed'] = test['bbox'].apply(ast.literal_eval)
test = add_camera_cols(test)

print(f'Train2: {len(train2):,} instances  |  {train2["image_id"].nunique():,} images')
print(f'Test:   {len(test):,} instances   |  {test["image_id"].nunique():,} images')
print('\nClass distribution (train2):')
print(train2['class_name'].value_counts().to_string())
print('\nCamera distribution (train2):')
print(train2.groupby('camera')['image_id'].count().sort_values(ascending=False).to_string())
print('\nCamera distribution (test):')
print(test.groupby('camera')['image_id'].count().sort_values(ascending=False).to_string())

In [ ]:
# ── Image loading and cropping utilities ────────────────────────────────────
def load_image(image_id, source):
    folder = {'train2': TRAIN2_IMGS, 'test': TEST_IMGS}[source]
    return Image.open(folder / image_id).convert('RGB')


def crop_with_padding(image, bbox, padding=0.12, make_square=True):
    """Crop pig instance from image using bounding box [x, y, w, h] with padding."""
    img_w, img_h = image.size
    x, y, w, h   = map(float, bbox)
    x1 = x - w * padding;      y1 = y - h * padding
    x2 = x + w + w * padding;  y2 = y + h + h * padding
    if make_square:
        side   = max(x2 - x1, y2 - y1)
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        x1, x2 = cx - side / 2, cx + side / 2
        y1, y2 = cy - side / 2, cy + side / 2
    x1 = max(0, int(round(x1)));      y1 = max(0, int(round(y1)))
    x2 = min(img_w, int(round(x2)));  y2 = min(img_h, int(round(y2)))
    return image.crop((max(x1, 0), max(y1, 0), max(x2, x1 + 1), max(y2, y1 + 1)))

In [ ]:
# ── Camera-specific augmentation functions ───────────────────────────────────
#
# Each function transforms a train-camera crop to simulate the corresponding
# test-camera appearance (brightness, colour, mirroring).
# Cameras with horizontal flip require updating Lateral_lying_left/right labels.

FLIP_LABEL_MAP = {0: 1, 1: 0, 2: 2, 3: 3, 4: 4}  # left ↔ right after hflip


def aug_pen2_tur_cam1(img):
    """Horizontal flip + slight colour cooling (saturation ↓, brightness ↓ slightly)."""
    img = TF.hflip(img)
    img = TF.adjust_saturation(img, saturation_factor=0.8)
    img = TF.adjust_brightness(img, brightness_factor=0.95)
    return img


def aug_pen1_tur_cam2(img):
    """Horizontal flip + brightness increase + slight desaturation."""
    img = TF.hflip(img)
    img = TF.adjust_brightness(img, brightness_factor=1.35)
    img = TF.adjust_saturation(img, saturation_factor=0.9)
    return img


def aug_pen2_orb_cam1(img):
    """Horizontal flip + darkening + contrast boost + greenish shift + partial greyscale."""
    img = TF.hflip(img)
    img = TF.adjust_brightness(img, brightness_factor=0.6)
    img = TF.adjust_contrast(img, contrast_factor=1.5)
    img_np = np.array(img).astype(float)
    img_np[:, :, 0] = (img_np[:, :, 0] * 0.85).clip(0, 255)  # less red
    img_np[:, :, 1] = (img_np[:, :, 1] * 1.10).clip(0, 255)  # more green
    img_np[:, :, 2] = (img_np[:, :, 2] * 0.80).clip(0, 255)  # less blue
    img_shifted = Image.fromarray(img_np.clip(0, 255).astype(np.uint8))
    img_grey    = TF.to_grayscale(img_shifted, num_output_channels=3)
    grey_np     = np.array(img_grey).astype(float)
    blended     = (0.65 * img_np + 0.35 * grey_np).clip(0, 255).astype(np.uint8)
    return Image.fromarray(blended)


def aug_pen2_tur_cam2(img):
    """Brightness increase + slight desaturation (no flip)."""
    img = TF.adjust_brightness(img, brightness_factor=1.3)
    img = TF.adjust_saturation(img, saturation_factor=0.85)
    return img


def aug_pen2_orb_cam2(img):
    """Darkening + contrast boost + partial greyscale blend (no flip)."""
    img = TF.adjust_brightness(img, brightness_factor=0.60)
    img = TF.adjust_contrast(img, contrast_factor=1.5)
    img_grey = TF.to_grayscale(img, num_output_channels=3)
    img_np   = np.array(img).astype(float)
    grey_np  = np.array(img_grey).astype(float)
    blended  = (0.65 * img_np + 0.35 * grey_np).clip(0, 255).astype(np.uint8)
    return Image.fromarray(blended)


# (pen1_orb_cam1, pen1_orb_cam2 have no test-camera counterpart — no augmentation)
CAMERA_AUG_FN = {
    'pen2_tur_cam1': (aug_pen2_tur_cam1, True),   # flip → label correction needed
    'pen1_tur_cam2': (aug_pen1_tur_cam2, True),
    'pen2_orb_cam1': (aug_pen2_orb_cam1, True),
    'pen2_tur_cam2': (aug_pen2_tur_cam2, False),  # no flip
    'pen2_orb_cam2': (aug_pen2_orb_cam2, False),
}

print('Camera augmentation functions defined ✓')
for cam, (fn, flip) in CAMERA_AUG_FN.items():
    print(f'  {cam:20s}  flip_label={flip}')

In [ ]:
# ── Transforms ───────────────────────────────────────────────────────────────
class AddGaussianNoise:
    """Add small Gaussian noise to a normalised tensor with probability p."""
    def __init__(self, std=0.02, p=0.15):
        self.std, self.p = std, p

    def __call__(self, tensor):
        if torch.rand(1).item() < self.p:
            tensor = torch.clamp(
                tensor + torch.randn_like(tensor) * self.std, 0., 1.)
        return tensor


# Applied after camera-specific PIL augmentation during supervised training
BASE_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    AddGaussianNoise(std=0.02, p=0.15),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Used for test inference and pseudo-label generation (no random noise)
VAL_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Applied to pseudo-labeled test crops during SSL fine-tuning.
# General augmentation only — no camera-specific transforms since
# test images are already captured by their natural cameras.
SSL_TRANSFORM = transforms.Compose([
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05),
    transforms.RandomRotation(degrees=10),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    AddGaussianNoise(std=0.02, p=0.15),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print('Transforms defined ✓')

In [ ]:
# ── Dataset classes ──────────────────────────────────────────────────────────

class PigDatasetCameraAug(Dataset):
    """
    Supervised training dataset.

    For each observation the original crop is always included.
    When the camera has a registered augmentation, a second (augmented) copy
    is added.  For cameras whose augmentation includes a horizontal flip,
    the label is remapped via FLIP_LABEL_MAP so that
    Lateral_lying_left ↔ Lateral_lying_right remains correct.
    """
    def __init__(self, df, camera_aug_fn, base_transform, is_train=True):
        self.base_transform = base_transform
        self.camera_aug_fn  = camera_aug_fn
        self.is_train       = is_train

        df = df.reset_index(drop=True)
        self.df = df

        # Each entry: (df_iloc_index, do_aug, label)
        self.samples = []
        for idx, row in df.iterrows():
            cam   = row.get('camera', 'unknown')
            label = int(row['class_id'])
            self.samples.append((idx, False, label))
            if is_train and cam in camera_aug_fn:
                _, flip_label = camera_aug_fn[cam]
                new_label = FLIP_LABEL_MAP[label] if flip_label else label
                self.samples.append((idx, True, new_label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        row_idx, do_aug, label = self.samples[idx]
        row   = self.df.iloc[row_idx]
        image = load_image(row['image_id'], row['source'])
        crop  = crop_with_padding(image, row['bbox_parsed'], padding=0.12, make_square=True)
        if do_aug:
            aug_fn, _ = self.camera_aug_fn[row['camera']]
            crop = aug_fn(crop)
        crop = self.base_transform(crop)
        return crop, label


class PigTestDataset(Dataset):
    """Test dataset — returns (image_tensor, row_id) for submission."""
    def __init__(self, df, transform):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = load_image(row['image_id'], row['source'])
        crop  = crop_with_padding(image, row['bbox_parsed'], padding=0.12, make_square=True)
        return self.transform(crop), str(row['row_id'])


class PseudoLabelDataset(Dataset):
    """
    Dataset for pseudo-labeled test instances used during SSL fine-tuning.

    The DataFrame must have columns: image_id, source, bbox_parsed, class_id (pseudo-label).
    SSL_TRANSFORM is applied (general augmentation, no camera-specific transformations
    because test images are already from their natural cameras).
    """
    def __init__(self, df, transform):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = load_image(row['image_id'], row['source'])
        crop  = crop_with_padding(image, row['bbox_parsed'], padding=0.12, make_square=True)
        return self.transform(crop), int(row['class_id'])


print('Dataset classes defined ✓')

In [ ]:
# ── Weighted DataLoader builder ───────────────────────────────────────────────
def build_weighted_loader(dataset, labels, batch_size, num_workers=2):
    """
    Create a DataLoader with WeightedRandomSampler to address class imbalance.

    Args:
        dataset:  torch Dataset
        labels:   list/array of integer class labels matching dataset order
        batch_size, num_workers: DataLoader kwargs
    """
    class_counts  = np.bincount(labels, minlength=NUM_CLASSES)
    class_weights = 1.0 / np.maximum(class_counts, 1)
    sample_weights = np.array([class_weights[l] for l in labels])

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=True,
    )


# ── Build initial supervised training loader ─────────────────────────────────
train_dataset = PigDatasetCameraAug(
    train2, CAMERA_AUG_FN, BASE_TRANSFORM, is_train=True
)
train_labels = [s[2] for s in train_dataset.samples]
train_loader = build_weighted_loader(train_dataset, train_labels, BATCH_SIZE)

class_counts_aug = np.bincount(train_labels, minlength=NUM_CLASSES)
print(f'Train2 instances:           {len(train2):,}')
print(f'After camera augmentation:  {len(train_dataset):,}  '
      f'(×{len(train_dataset)/len(train2):.2f})')
print(f'Train batches:              {len(train_loader)}')
print('\nClass distribution after augmentation:')
for cid, cnt in enumerate(class_counts_aug):
    print(f'  {CLASS_NAMES[cid]:25s}: {cnt:,}')

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────────
def create_model():
    model = models.efficientnet_v2_s(
        weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
    )
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    return model.to(DEVICE)


model = create_model()
print(f'EfficientNetV2-S  |  parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Training utilities ───────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, all_preds, all_targets = 0.0, [], []
    n_batches = len(loader)

    for batch_i, (images, labels) in enumerate(loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        all_preds.extend(outputs.argmax(dim=1).detach().cpu().tolist())
        all_targets.extend(labels.cpu().tolist())

        filled = int(30 * (batch_i + 1) / n_batches)
        bar    = '█' * filled + '░' * (30 - filled)
        print(f'\r  [train] |{bar}| {100*(batch_i+1)/n_batches:5.1f}%  '
              f'{batch_i+1}/{n_batches}  loss={loss.item():.4f}',
              end='', flush=True)

    print()
    n = len(loader.dataset)
    return (
        total_loss / n,
        accuracy_score(all_targets, all_preds),
        f1_score(all_targets, all_preds, average='macro'),
    )


def plot_history(history, title='Training Curves', save_name=None):
    hist_df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].plot(hist_df['epoch'], hist_df['train_loss'], marker='o', markersize=4)
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=0.3)
    best_f1 = hist_df['train_f1'].max()
    axes[1].plot(hist_df['epoch'], hist_df['train_f1'], marker='o', markersize=4, color='green')
    axes[1].axhline(best_f1, color='red', linestyle='--', linewidth=1,
                    label=f'best = {best_f1:.4f}')
    axes[1].set_title('Train Macro-F1'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    if save_name:
        plt.savefig(save_name, dpi=120, bbox_inches='tight')
    plt.show()
    return best_f1


print('Training utilities defined ✓')

## Phase 1 — Supervised Training on train2

In [ ]:
print('=' * 65)
print('PHASE 1: Supervised training on train2 with camera augmentations')
print('=' * 65)

criterion_p1 = nn.CrossEntropyLoss()
optimizer_p1 = torch.optim.Adam(model.parameters(), lr=LR_PHASE1)
scheduler_p1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p1, T_max=EPOCHS_PHASE1, eta_min=1e-6
)

best_f1_p1, best_state_p1, history_p1 = 0.0, None, []

for epoch in range(EPOCHS_PHASE1):
    if epoch % 5 == 0 and DEVICE.type == 'cuda':
        _x = torch.randn(10000, 10000, device=DEVICE)
        _t = time.time(); _ = _x @ _x; torch.cuda.synchronize()
        _e = time.time() - _t
        print(f'  [GPU check] {_e:.3f}s  {"✓" if _e < 1.0 else "⚠"}')
        del _x; torch.cuda.empty_cache()

    t0 = time.time()
    tr_loss, tr_acc, tr_f1 = train_one_epoch(
        model, train_loader, optimizer_p1, criterion_p1
    )
    scheduler_p1.step()
    elapsed = time.time() - t0

    history_p1.append({'epoch': epoch + 1,
                       'train_loss': tr_loss, 'train_acc': tr_acc, 'train_f1': tr_f1})

    flag = '  ← best' if tr_f1 > best_f1_p1 else ''
    print(f'Epoch {epoch+1:>2}/{EPOCHS_PHASE1}  ({elapsed/60:.1f} min)  '
          f'loss {tr_loss:.4f}  acc {tr_acc:.4f}  f1 {tr_f1:.4f}  '
          f'lr {optimizer_p1.param_groups[0]["lr"]:.1e}{flag}', flush=True)

    if tr_f1 > best_f1_p1:
        best_f1_p1    = tr_f1
        best_state_p1 = copy.deepcopy(model.state_dict())
        torch.save({
            'model_state_dict': best_state_p1,
            'class_names':      CLASS_NAMES,
            'best_train_f1':    best_f1_p1,
            'epoch':            epoch + 1,
            'phase':            'phase1',
        }, SAVE_PATH_PHASE1)
        print(f'  Saved → {SAVE_PATH_PHASE1}', flush=True)

print(f'\nPhase 1 best train macro-F1: {best_f1_p1:.4f}')

In [ ]:
plot_history(history_p1, title='Phase 1: Supervised Training', save_name='phase1_curves.png')

## Phase 2 — Generate Pseudo-Labels for Test Set

The model trained in Phase 1 is used to produce soft predictions for every test instance.  
Only instances whose maximum softmax probability exceeds the confidence threshold are kept as pseudo-labels for the SSL fine-tuning step.

In [ ]:
# ── Load best Phase 1 checkpoint ────────────────────────────────────────────
ckpt = torch.load(SAVE_PATH_PHASE1, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded Phase 1 model  (F1={ckpt["best_train_f1"]:.4f}, epoch={ckpt["epoch"]})')


def generate_pseudo_labels(model, test_df, transform, confidence_threshold):
    """
    Run inference on test_df and return two DataFrames:
      - full_pseudo_df : all test instances with 'class_id' and 'confidence' columns
      - high_conf_df   : subset with confidence >= confidence_threshold

    The returned DataFrames retain all original columns from test_df so they
    can be used directly as PseudoLabelDataset input.
    """
    dataset = PigTestDataset(test_df, transform)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

    all_preds, all_confs = [], []

    model.eval()
    with torch.no_grad():
        for batch_i, (images, _) in enumerate(loader):
            images = images.to(DEVICE)
            probs  = F.softmax(model(images), dim=1)
            confs, preds = probs.max(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_confs.extend(confs.cpu().tolist())
            filled = int(30 * (batch_i + 1) / len(loader))
            bar    = '█' * filled + '░' * (30 - filled)
            print(f'\r  [pseudo-label] |{bar}| {batch_i+1}/{len(loader)}',
                  end='', flush=True)
    print()

    full_pseudo_df = test_df.copy().reset_index(drop=True)
    full_pseudo_df['class_id']   = all_preds   # pseudo-label
    full_pseudo_df['confidence'] = all_confs

    high_conf_df = full_pseudo_df[
        full_pseudo_df['confidence'] >= confidence_threshold
    ].copy()

    print(f'Threshold {confidence_threshold:.2f}: '
          f'{len(high_conf_df):,} / {len(full_pseudo_df):,} '
          f'({100*len(high_conf_df)/len(full_pseudo_df):.1f}%) high-confidence samples')
    print('Pseudo-label class distribution (high-confidence):')
    for cid, cnt in sorted(Counter(high_conf_df['class_id'].tolist()).items()):
        print(f'  [{cid}] {CLASS_NAMES[cid]:25s}: {cnt:,}  '
              f'({100*cnt/max(len(high_conf_df),1):.1f}%)')

    return full_pseudo_df, high_conf_df


pseudo_all_r1, pseudo_high_r1 = generate_pseudo_labels(
    model, test, VAL_TRANSFORM, confidence_threshold=SSL_CONF_THRESH[0]
)

# Confidence histogram
plt.figure(figsize=(10, 4))
plt.hist(pseudo_all_r1['confidence'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
plt.axvline(SSL_CONF_THRESH[0], color='red', linestyle='--',
            label=f'Round 1 threshold = {SSL_CONF_THRESH[0]}')
plt.axvline(SSL_CONF_THRESH[1], color='orange', linestyle='--',
            label=f'Round 2 threshold = {SSL_CONF_THRESH[1]}')
plt.xlabel('Prediction Confidence'); plt.ylabel('Count')
plt.title('Test Set Prediction Confidence Distribution (Phase 1 model)')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig('confidence_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## Phase 3 — Iterative SSL Fine-Tuning

**Round 1** — high-confidence pseudo-labels (threshold = 0.90)  
**Round 2** — updated pseudo-labels with lower threshold (0.85) from the Round-1 model

Each round combines supervised `train2` (with camera augmentations) and pseudo-labeled test images (with general augmentation) and uses a lower learning rate than Phase 1 to preserve learned features.

In [ ]:
def run_ssl_round(model, train2_df, pseudo_df, epochs, lr, round_num):
    """
    Fine-tune model on combined supervised + pseudo-labeled data.

    Args:
        model:      current model (will be updated in-place with best weights)
        train2_df:  labelled training DataFrame
        pseudo_df:  high-confidence pseudo-labelled test DataFrame
        epochs:     fine-tuning epochs
        lr:         peak learning rate for cosine schedule
        round_num:  used for display only

    Returns:
        model (loaded with best weights from this round), best_f1, history
    """
    print(f'\n{"="*65}')
    print(f'SSL Round {round_num}  |  pseudo-labels: {len(pseudo_df):,}  '
          f'|  lr={lr}  |  epochs={epochs}')
    print(f'{"="*65}')

    # Supervised dataset (with camera augmentations)
    sup_ds    = PigDatasetCameraAug(train2_df, CAMERA_AUG_FN, BASE_TRANSFORM, is_train=True)
    # Pseudo-labeled test dataset (general augmentation)
    pseudo_ds = PseudoLabelDataset(pseudo_df, SSL_TRANSFORM)
    # Combined
    combined_ds = ConcatDataset([sup_ds, pseudo_ds])

    sup_labels    = [s[2] for s in sup_ds.samples]
    pseudo_labels = pseudo_df['class_id'].astype(int).tolist()
    all_labels    = sup_labels + pseudo_labels

    print(f'  Supervised samples (with cam aug): {len(sup_ds):,}')
    print(f'  Pseudo-labeled samples:            {len(pseudo_ds):,}')
    print(f'  Total combined:                    {len(combined_ds):,}')
    print('  Combined class distribution:')
    combined_counts = np.bincount(all_labels, minlength=NUM_CLASSES)
    for cid, cnt in enumerate(combined_counts):
        print(f'    [{cid}] {CLASS_NAMES[cid]:25s}: {cnt:,}')

    combined_loader = build_weighted_loader(
        combined_ds, all_labels, BATCH_SIZE
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=lr / 10
    )

    best_f1, best_state, history = 0.0, None, []

    for epoch in range(epochs):
        t0 = time.time()
        tr_loss, tr_acc, tr_f1 = train_one_epoch(
            model, combined_loader, optimizer, criterion
        )
        scheduler.step()
        elapsed = time.time() - t0

        history.append({'epoch': epoch + 1,
                        'train_loss': tr_loss, 'train_acc': tr_acc, 'train_f1': tr_f1})

        flag = '  ← best' if tr_f1 > best_f1 else ''
        print(f'  Round {round_num} Epoch {epoch+1:>2}/{epochs}  ({elapsed/60:.1f} min)  '
              f'loss {tr_loss:.4f}  acc {tr_acc:.4f}  f1 {tr_f1:.4f}  '
              f'lr {optimizer.param_groups[0]["lr"]:.1e}{flag}', flush=True)

        if tr_f1 > best_f1:
            best_f1    = tr_f1
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    print(f'  Round {round_num} best macro-F1: {best_f1:.4f}')
    plot_history(history,
                 title=f'SSL Round {round_num} Fine-tuning',
                 save_name=f'ssl_round{round_num}_curves.png')
    return model, best_f1, history


print('SSL round function defined ✓')

In [ ]:
# ── SSL Round 1 ──────────────────────────────────────────────────────────────
model, ssl_f1_r1, history_ssl1 = run_ssl_round(
    model, train2, pseudo_high_r1,
    epochs=EPOCHS_SSL, lr=LR_SSL[0], round_num=1
)

In [ ]:
# ── SSL Round 2: re-generate pseudo-labels with updated model ────────────────
print('Generating updated pseudo-labels with Round 1 model...')
pseudo_all_r2, pseudo_high_r2 = generate_pseudo_labels(
    model, test, VAL_TRANSFORM, confidence_threshold=SSL_CONF_THRESH[1]
)

# Confidence shift plot (Round 1 vs Round 2 model)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, pall, title in [
    (axes[0], pseudo_all_r1, f'Phase 1 model (thresh={SSL_CONF_THRESH[0]})'),
    (axes[1], pseudo_all_r2, f'Round 1 model (thresh={SSL_CONF_THRESH[1]})'),
]:
    ax.hist(pall['confidence'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ax.set_title(title); ax.set_xlabel('Confidence'); ax.grid(alpha=0.3)
plt.suptitle('Confidence Distribution Shift Across SSL Rounds', fontweight='bold')
plt.tight_layout()
plt.savefig('confidence_shift.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── SSL Round 2 fine-tuning ──────────────────────────────────────────────────
model, ssl_f1_r2, history_ssl2 = run_ssl_round(
    model, train2, pseudo_high_r2,
    epochs=EPOCHS_SSL, lr=LR_SSL[1], round_num=2
)

In [ ]:
# ── Save final SSL model ─────────────────────────────────────────────────────
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names':      CLASS_NAMES,
    'phase1_f1':        best_f1_p1,
    'ssl_round1_f1':    ssl_f1_r1,
    'ssl_round2_f1':    ssl_f1_r2,
    'phase':            'ssl_final',
}, SAVE_PATH_SSL)
print(f'Final SSL model saved → {SAVE_PATH_SSL}')
print(f'  Phase 1 best F1:   {best_f1_p1:.4f}')
print(f'  SSL Round 1 F1:    {ssl_f1_r1:.4f}')
print(f'  SSL Round 2 F1:    {ssl_f1_r2:.4f}')

from google.colab import files
files.download(SAVE_PATH_SSL)
print('Model downloaded ✓')

## Final Inference & Submission

In [ ]:
print('=' * 65)
print('FINAL INFERENCE: generating submission CSV')
print('=' * 65)

# Load final SSL checkpoint (in case the kernel was re-run from this cell)
ckpt_final = torch.load(SAVE_PATH_SSL, map_location=DEVICE)
model.load_state_dict(ckpt_final['model_state_dict'])
model.eval()
print(f'Model loaded  |  SSL Round 2 F1: {ckpt_final["ssl_round2_f1"]:.4f}')

test_loader = DataLoader(
    PigTestDataset(test, VAL_TRANSFORM),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

all_row_ids, all_preds = [], []

with torch.no_grad():
    for batch_i, (images, row_ids) in enumerate(test_loader):
        outputs = model(images.to(DEVICE))
        preds   = outputs.argmax(dim=1).cpu().tolist()
        all_row_ids.extend(list(row_ids))
        all_preds.extend(preds)
        filled = int(30 * (batch_i + 1) / len(test_loader))
        bar    = '█' * filled + '░' * (30 - filled)
        print(f'\r  [inference] |{bar}| {batch_i+1}/{len(test_loader)}',
              end='', flush=True)

print(f'\nInference complete: {len(all_preds):,} predictions')

# Build and validate submission DataFrame
submission = pd.DataFrame({'row_id': all_row_ids, 'class_id': all_preds})
sample_sub = pd.read_csv(BASE_DIR / 'sample_submission.csv')

assert set(submission['row_id']) == set(sample_sub['row_id']), \
    'row_id mismatch with sample_submission!'
assert submission['class_id'].between(0, 4).all(), \
    'class_id out of range [0, 4]!'

submission = (
    submission.set_index('row_id')
              .reindex(sample_sub['row_id'])
              .reset_index()
)

print('\nPrediction distribution:')
for cid, cnt in sorted(submission['class_id'].value_counts().items()):
    print(f'  [{cid}] {CLASS_NAMES[cid]:25s}: {cnt:,}  ({100*cnt/len(submission):.1f}%)')

submission.to_csv(SUBMISSION_PATH, index=False)
print(f'\nSaved → {SUBMISSION_PATH}')
print(submission.head(10).to_string())

files.download(SUBMISSION_PATH)
print('Submission downloaded ✓')

In [ ]:
# ── Training summary plot ────────────────────────────────────────────────────
def offset_history(history, offset):
    h = pd.DataFrame(history)
    h['epoch'] = h['epoch'] + offset
    return h

h1 = pd.DataFrame(history_p1)
h2 = offset_history(history_ssl1, EPOCHS_PHASE1)
h3 = offset_history(history_ssl2, EPOCHS_PHASE1 + EPOCHS_SSL)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = ['royalblue', 'darkorange', 'green']
labels = ['Phase 1 (supervised)', 'SSL Round 1', 'SSL Round 2']

for ax_idx, (ax, col) in enumerate(zip(axes, ['train_loss', 'train_f1'])):
    for hist, color, label in zip([h1, h2, h3], colors, labels):
        ax.plot(hist['epoch'], hist[col], marker='o', markersize=4,
                color=color, label=label)
    # phase boundaries
    ax.axvline(EPOCHS_PHASE1, color='gray', linestyle=':', alpha=0.7)
    ax.axvline(EPOCHS_PHASE1 + EPOCHS_SSL, color='gray', linestyle=':', alpha=0.7)
    ax.set_xlabel('Epoch'); ax.grid(alpha=0.3); ax.legend()

axes[0].set_title('Loss')
axes[1].set_title('Train Macro-F1')
plt.suptitle('Full Training Summary — EfficientNetV2-S + SSL',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('full_training_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Phase 1 best F1:   {best_f1_p1:.4f}')
print(f'SSL Round 1 F1:    {ssl_f1_r1:.4f}')
print(f'SSL Round 2 F1:    {ssl_f1_r2:.4f}')